In [8]:
CHUNK_MIN_LEN = 80
TFIDF_TOP_K = 40
FINAL_TOP_K = 8

ALPHA_TFIDF = 0.6
ALPHA_SEMANTIC = 0.25
ALPHA_PAGERANK = 0.15

SIMILARITY_THRESHOLD = 0.65
PAGERANK_DAMPING = 0.85

EMBED_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
GEN_MODEL_NAME = "google/flan-t5-base"

OUTPUT_DIR = "handbook-1.0"

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

GEN_MODEL_NAME = "google/flan-t5-base"
OUTPUT_DIR = "handbook-1.0"

tokenizer = AutoTokenizer.from_pretrained(GEN_MODEL_NAME)
model = AutoModelForSeq2SeqLM.from_pretrained(GEN_MODEL_NAME)

tokenizer.save_pretrained(OUTPUT_DIR)
model.save_pretrained(OUTPUT_DIR)

In [ ]:
import pickle, numpy as np
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import re

class HandbookPipeline:
    def __init__(self, path):
        with open(f"{path}/chunks.pkl", "rb") as f:
            self.chunks = pickle.load(f)

        with open(f"{path}/tfidf.pkl", "rb") as f:
            self.vectorizer, self.tfidf_matrix = pickle.load(f)

        self.embed_model = SentenceTransformer(f"{path}/embed_model")

        self.tokenizer = AutoTokenizer.from_pretrained(GEN_MODEL_NAME)
        self.generator = AutoModelForSeq2SeqLM.from_pretrained(GEN_MODEL_NAME)

    def clean_query(self, q):
        q = q.lower()
        q = re.sub(r'[^a-z0-9\s]', '', q)
        return q

    def retrieve(self, query):
        query = self.clean_query(query)

        q_vec = self.vectorizer.transform([query])
        tfidf_scores = (self.tfidf_matrix @ q_vec.T).toarray().flatten()
        idxs = tfidf_scores.argsort()[::-1][:TFIDF_TOP_K]

        q_emb = self.embed_model.encode([query], normalize_embeddings=True)[0]

        scored = []
        for i in idxs:
            c = self.chunks[i]

            sem = float(np.dot(q_emb, c["embedding"]))
            tfidf = tfidf_scores[i]
            pr = c["pagerank"]

            score = (
                ALPHA_TFIDF * tfidf +
                ALPHA_SEMANTIC * sem +
                ALPHA_PAGERANK * pr
            )

            scored.append((c, score))

        scored = sorted(scored, key=lambda x: x[1], reverse=True)

        return [c for c, _ in scored[:FINAL_TOP_K]]

    def generate(self, query, docs):
        context = "\n\n".join([d["text"] for d in docs])
        prompt = f"Answer strictly from context.\n\n{context}\n\nQuestion: {query}\nAnswer:"
        inputs = self.tokenizer(prompt, return_tensors="pt", truncation=True, max_length=1024)
        outputs = self.generator.generate(**inputs, max_new_tokens=200)
        return self.tokenizer.decode(outputs[0], skip_special_tokens=True)

    def ask(self, query):
        docs = self.retrieve(query)
        answer = self.generate(query, docs)
        return answer, docs

In [10]:
pipeline = HandbookPipeline("handbook-1.0")

answer, docs = pipeline.ask("What is the minimum GPA requirement?")
print(answer)

Loading weights: 100%|██████████| 282/282 [00:00<00:00, 23948.53it/s]
The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.
Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.59it/s]


2.75 for Engineering, IT, CS, Natural & Applied Biosciences, Architecture & Industrial Design and 3.00 for UG degree programmes of NBS & S3H
